In [4]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Tuple

import numpy as np
import pandas as pd

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [5]:


(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

# Normalize data (scale pixel values between 0 and 1)
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0 

print("Train:", x_train.shape, y_train.shape)

print("Test :", x_test.shape, y_test.shape)

 

class_names = ["airplane","automobile","bird","cat","deer","dog","frog","horse","ship","truck"]


170498071/170498071 [==============================] - 8s 0us/step
Train: (50000, 32, 32, 3) (50000, 1)
Test : (10000, 32, 32, 3) (10000, 1)


In [6]:
y_train_va=y_train.copy()
#vehicle=0, animal =1
#if y_train[i] is 0,1,8,9 the y_train_va[i]=0 otherwise y_train_va[i]=1
for i in range(len(y_train_va)):
    if y_train_va[i][0] in [0, 1, 8, 9]:
        y_train_va[i]=0
    else:
        y_train_va[i]=1

y_test_va=y_test.copy()
for i in range(len(y_test_va)):
    if y_test_va[i][0] in [0, 1, 8, 9]:
        y_test_va[i]=0
    else:
        y_test_va[i]=1


In [10]:

def build_multihead_cnn(input_dim:tuple[int,int,int],lr:float=1e-3)->tf.keras.Model:
    """Builds a multihead CNN model: one model predicts the class of the image, the other model predicts if it is a vehicle or animal"""
    with tf.device('/GPU:0'):
        inputs=tf.keras.Input(shape=input_dim)
        x=keras.layers.Conv2D(32, (3,3), activation='relu', padding='same')(inputs)   #image size 32x32
        x=keras.layers.MaxPooling2D((2,2))(x)                                       #image size 16x16
        x=keras.layers.Conv2D(64, (3,3), activation='relu')(x)                  
        x=keras.layers.MaxPooling2D((2,2))(x)                                       #image size 8x8
        x=keras.layers.Conv2D(128, (3,3), activation='relu')(x)
        x=keras.layers.MaxPooling2D((2,2))(x)                                       #image size 4x4
        x=keras.layers.Flatten()(x)
        x=keras.layers.Dense(128, activation='relu')(x)
        x=keras.layers.Dense(32, activation='softmax')(x)
        shared=tf.keras.layers.Dense(16, activation='relu', name="shared_repr")(x)

        class_output=tf.keras.layers.Dense(10, activation='softmax', name="class_output")(shared)
        class_vehicle_animal=tf.keras.layers.Dense(2, activation='softmax', name="class_vehicle_animal")(shared)

        model=tf.keras.Model(inputs,outputs={"class_output":class_output,"class_vehicle_animal":class_vehicle_animal})
        model.compile(
            optimizer='adam', 
            loss={"class_output":"sparse_categorical_crossentropy", "class_vehicle_animal":'sparse_categorical_crossentropy',
            },
            #Loss weights starts with 1.0/1.0
            loss_weights={"class_output":1.0, "class_vehicle_animal":1.0}, 
            metrics={
                "class_output": ['accuracy'],
                "class_vehicle_animal": ['accuracy']
            })
    return model
    
multihead_cnn=build_multihead_cnn((32,32,3),lr=1e-3)
multihead_cnn.summary()
    
    

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_3 (InputLayer)           [(None, 32, 32, 3)]  0           []                               
                                                                                                  
 conv2d_3 (Conv2D)              (None, 32, 32, 32)   896         ['input_3[0][0]']                
                                                                                                  
 max_pooling2d_3 (MaxPooling2D)  (None, 16, 16, 32)  0           ['conv2d_3[0][0]']               
                                                                                                  
 conv2d_4 (Conv2D)              (None, 14, 14, 64)   18496       ['max_pooling2d_3[0][0]']        
                                                                                            

In [ ]:
multihead_cnn.fit(
    x_train,
    {
        "class_output": y_train,
        "class_vehicle_animal": y_train_va
    },
    epochs=30,
    batch_size=32,
    verbose=1,
    validation_data=(x_test, {
        "class_output": y_test,
        "class_vehicle_animal": y_test_va
    })
)



Epoch 1/30
1563/1563 [==============================] - 11s 7ms/step - loss: 2.9765 - class_output_loss: 2.3031 - class_vehicle_animal_loss: 0.6734 - class_output_accuracy: 0.1014 - class_vehicle_animal_accuracy: 0.5996 - val_loss: 2.9760 - val_class_output_loss: 2.3029 - val_class_vehicle_animal_loss: 0.6731 - val_class_output_accuracy: 0.1000 - val_class_vehicle_animal_accuracy: 0.6000
Epoch 2/30
1563/1563 [==============================] - 11s 7ms/step - loss: 2.9762 - class_output_loss: 2.3030 - class_vehicle_animal_loss: 0.6732 - class_output_accuracy: 0.0987 - class_vehicle_animal_accuracy: 0.6000 - val_loss: 2.9758 - val_class_output_loss: 2.3028 - val_class_vehicle_animal_loss: 0.6731 - val_class_output_accuracy: 0.1000 - val_class_vehicle_animal_accuracy: 0.6000
Epoch 3/30
1563/1563 [==============================] - 11s 7ms/step - loss: 2.9761 - class_output_loss: 2.3029 - class_vehicle_animal_loss: 0.6731 - class_output_accuracy: 0.0972 - class_vehicle_animal_accuracy: 0.600

In [7]:

# test the multi on test data
multihead_cnn.evaluate(x_test, {
    "class_output": y_test,
    "class_vehicle_animal": y_test_va
})




313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - class_output_accuracy: 0.6743 - class_output_loss: 1.2608 - class_vehicle_animal_accuracy: 0.9385 - class_vehicle_animal_loss: 0.2615 - loss: 1.5239


[1.523878812789917,
 1.2607707977294922,
 0.2615327835083008,
 0.6743000149726868,
 0.9384999871253967]

In [1]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
# Check if TensorFlow is built with CUDA support
print("Built with CUDA:", tf.test.is_built_with_cuda())

# Check for GPU devices
gpu_devices = tf.config.list_physical_devices('GPU')

if not gpu_devices:
    print("No GPU found. Troubleshooting steps:")
    print("1. Ensure NVIDIA drivers are installed.")
    print("2. Ensure CUDA Toolkit and cuDNN are installed and in your PATH.")
    print("3. Note: TensorFlow versions > 2.10 do not support GPU on native Windows. Use WSL2 or install tensorflow==2.10.0.")
else:
    # Enable memory growth to prevent TF from grabbing all VRAM at once
    for gpu in gpu_devices:
        tf.config.experimental.set_memory_growth(gpu, True)
    print("GPUs available:", gpu_devices)

TensorFlow version: 2.10.0
Built with CUDA: True
GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [9]:
print(tf.config.list_physical_devices("GPU"))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
